# 부록 05. StateGraph API로 커스텀 그래프 구축하기

LangGraph의 `StateGraph` API를 사용하면 세밀하게 제어되는 에이전트 워크플로우를 구축할 수 있습니다.

## 학습 목표
- StateGraph의 핵심 개념 (노드, 엣지, 상태) 이해
- 조건부 라우팅 구현 방법
- 실제 에이전트 워크플로우 구축

## 1. 환경 설정

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정'}")

## 2. StateGraph 기본 개념

StateGraph는 세 가지 핵심 요소로 구성됩니다:

1. **State (상태)**: 그래프를 통해 흐르는 데이터
2. **Nodes (노드)**: 상태를 처리하는 함수
3. **Edges (엣지)**: 노드 간의 연결

```
START → Node1 → Node2 → END
```

In [ ]:
from typing import TypedDict, Annotated, Literal
import operator
from langgraph.graph import StateGraph, START, END

# 1. 상태 정의 (TypedDict 사용 - LangGraph 1.0 필수)
class SimpleState(TypedDict):
    # Annotated와 operator.add를 사용하면 리스트에 항목이 추가됨
    messages: Annotated[list[str], operator.add]
    counter: int

# 2. 노드 함수 정의
def node_a(state: SimpleState) -> dict:
    """첫 번째 노드: 메시지 추가"""
    return {
        "messages": [f"Node A 처리 완료 (카운터: {state['counter']})"],
        "counter": state["counter"] + 1
    }

def node_b(state: SimpleState) -> dict:
    """두 번째 노드: 메시지 추가"""
    return {
        "messages": [f"Node B 처리 완료 (카운터: {state['counter']})"],
        "counter": state["counter"] + 1
    }

# 3. 그래프 구성
graph = StateGraph(SimpleState)

# 노드 추가
graph.add_node("node_a", node_a)
graph.add_node("node_b", node_b)

# 엣지 추가 (순차 실행)
graph.add_edge(START, "node_a")
graph.add_edge("node_a", "node_b")
graph.add_edge("node_b", END)

# 4. 그래프 컴파일
app = graph.compile()

print("그래프가 생성되었습니다.")

In [ ]:
# 그래프 실행
result = app.invoke({
    "messages": ["시작"],
    "counter": 0
})

print("=== 실행 결과 ===")
print(f"메시지: {result['messages']}")
print(f"최종 카운터: {result['counter']}")

## 3. 조건부 엣지 (Conditional Edges)

상태에 따라 다른 노드로 라우팅할 수 있습니다.

In [ ]:
class RouterState(TypedDict):
    query: str
    query_type: str
    result: str

def classify_query(state: RouterState) -> dict:
    """쿼리 유형 분류"""
    query = state["query"].lower()
    
    if "날씨" in query or "weather" in query:
        query_type = "weather"
    elif "계산" in query or "calculate" in query or any(c in query for c in "+-*/"):
        query_type = "calculation"
    else:
        query_type = "general"
    
    return {"query_type": query_type}

def handle_weather(state: RouterState) -> dict:
    """날씨 처리"""
    return {"result": f"날씨 정보: '{state['query']}'에 대한 날씨를 조회했습니다."}

def handle_calculation(state: RouterState) -> dict:
    """계산 처리"""
    return {"result": f"계산 결과: '{state['query']}'를 계산했습니다."}

def handle_general(state: RouterState) -> dict:
    """일반 처리"""
    return {"result": f"일반 응답: '{state['query']}'에 대한 답변입니다."}

# 라우팅 함수
def route_query(state: RouterState) -> Literal["weather", "calculation", "general"]:
    """쿼리 유형에 따라 라우팅"""
    return state["query_type"]

# 그래프 구성
router_graph = StateGraph(RouterState)

# 노드 추가
router_graph.add_node("classify", classify_query)
router_graph.add_node("weather", handle_weather)
router_graph.add_node("calculation", handle_calculation)
router_graph.add_node("general", handle_general)

# 엣지 추가
router_graph.add_edge(START, "classify")

# 조건부 엣지: classify 노드 이후 query_type에 따라 분기
router_graph.add_conditional_edges(
    "classify",
    route_query,
    {
        "weather": "weather",
        "calculation": "calculation",
        "general": "general"
    }
)

# 모든 핸들러는 END로
router_graph.add_edge("weather", END)
router_graph.add_edge("calculation", END)
router_graph.add_edge("general", END)

router_app = router_graph.compile()

print("라우터 그래프가 생성되었습니다.")

In [ ]:
# 다양한 쿼리 테스트
test_queries = [
    "오늘 서울 날씨 어때?",
    "15 + 27 계산해줘",
    "안녕하세요!"
]

for query in test_queries:
    result = router_app.invoke({"query": query, "query_type": "", "result": ""})
    print(f"쿼리: {query}")
    print(f"유형: {result['query_type']}")
    print(f"결과: {result['result']}")
    print("-" * 50)

## 4. LLM을 사용한 에이전트 그래프

실제 LLM과 도구를 사용하는 에이전트를 StateGraph로 구현해봅니다.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langgraph.prebuilt import ToolNode

# 도구 정의
@tool
def get_weather(city: str) -> str:
    """도시의 날씨를 조회합니다."""
    weather_db = {"서울": "맑음 22°C", "도쿄": "흐림 18°C", "뉴욕": "비 15°C"}
    return weather_db.get(city, f"{city}: 정보 없음")

@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다."""
    try:
        return str(eval(expression))
    except:
        return "계산 오류"

tools = [get_weather, calculate]

# 모델 설정 (도구 바인딩)
model = init_chat_model("gpt-4o-mini", temperature=0)
model_with_tools = model.bind_tools(tools)

print("모델과 도구가 준비되었습니다.")

In [ ]:
from typing import Sequence
from langchain_core.messages import BaseMessage

# 에이전트 상태 정의
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# 모델 호출 노드
def call_model(state: AgentState) -> dict:
    """LLM을 호출하여 응답 생성"""
    messages = state["messages"]
    response = model_with_tools.invoke(messages)
    return {"messages": [response]}

# 도구 실행 여부 결정
def should_continue(state: AgentState) -> Literal["tools", "end"]:
    """도구 호출이 필요한지 확인"""
    last_message = state["messages"][-1]
    
    # 도구 호출이 있으면 tools 노드로
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    # 없으면 종료
    return "end"

# 도구 노드 (LangGraph 내장)
tool_node = ToolNode(tools)

# 그래프 구성
agent_graph = StateGraph(AgentState)

# 노드 추가
agent_graph.add_node("agent", call_model)
agent_graph.add_node("tools", tool_node)

# 엣지 추가
agent_graph.add_edge(START, "agent")
agent_graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        "end": END
    }
)
agent_graph.add_edge("tools", "agent")  # 도구 실행 후 다시 에이전트로

# 컴파일
agent_app = agent_graph.compile()

print("에이전트 그래프가 생성되었습니다.")

In [ ]:
# 에이전트 테스트
test_messages = [
    "서울 날씨 알려줘",
    "123 * 456 계산해줘",
    "안녕하세요!"
]

for msg in test_messages:
    result = agent_app.invoke({"messages": [HumanMessage(content=msg)]})
    print(f"Q: {msg}")
    print(f"A: {result['messages'][-1].content}")
    print("-" * 50)

## 5. 그래프 시각화

In [ ]:
# 그래프 구조 출력 (ASCII)
try:
    print(agent_app.get_graph().draw_ascii())
except Exception as e:
    print(f"ASCII 시각화 실패: {e}")
    print("\n그래프 구조:")
    print("START → agent → (tools → agent) → END")
    print("                 ↑___________|")

## 6. Reducer 함수 이해하기

`Annotated`와 함께 사용하는 reducer 함수는 상태 업데이트 방식을 정의합니다.

In [ ]:
# 커스텀 reducer 예제
def merge_dicts(left: dict, right: dict) -> dict:
    """두 딕셔너리를 병합하는 reducer"""
    if left is None:
        return right
    if right is None:
        return left
    return {**left, **right}

class CustomState(TypedDict):
    # operator.add: 리스트에 항목 추가
    messages: Annotated[list, operator.add]
    
    # 커스텀 reducer: 딕셔너리 병합
    metadata: Annotated[dict, merge_dicts]
    
    # reducer 없음: 값 덮어쓰기
    current_step: str

def step1(state: CustomState) -> dict:
    return {
        "messages": ["Step 1 완료"],
        "metadata": {"step1_done": True},
        "current_step": "step1"
    }

def step2(state: CustomState) -> dict:
    return {
        "messages": ["Step 2 완료"],
        "metadata": {"step2_done": True},
        "current_step": "step2"
    }

# 그래프 구성
custom_graph = StateGraph(CustomState)
custom_graph.add_node("step1", step1)
custom_graph.add_node("step2", step2)
custom_graph.add_edge(START, "step1")
custom_graph.add_edge("step1", "step2")
custom_graph.add_edge("step2", END)

custom_app = custom_graph.compile()

# 실행
result = custom_app.invoke({
    "messages": ["시작"],
    "metadata": {"initial": True},
    "current_step": "init"
})

print("=== Reducer 동작 확인 ===")
print(f"messages (add): {result['messages']}")
print(f"metadata (merge): {result['metadata']}")
print(f"current_step (overwrite): {result['current_step']}")

## 7. 요약

### 이 노트북에서 배운 것

1. **StateGraph 기본 구성**
   - `StateGraph(State)`: 그래프 생성
   - `add_node(name, func)`: 노드 추가
   - `add_edge(from, to)`: 엣지 추가
   - `compile()`: 실행 가능한 앱으로 변환

2. **조건부 라우팅**
   - `add_conditional_edges(node, router_func, mapping)`
   - 상태에 따라 다른 노드로 분기

3. **상태 관리**
   - TypedDict로 상태 정의
   - Annotated + reducer로 업데이트 방식 지정

4. **LLM 에이전트 그래프**
   - 모델 노드 + 도구 노드 구성
   - ReAct 루프 구현

### 다음 단계
- 부록_06: Middleware로 에이전트 실행 제어하기